#### 1. Bronze processing

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/dhotepatil00@gmail.com/regis-healthcare/1_setup/utility

In [0]:
print(bronze_schema,silver_schema,gold_schema) 

In [0]:
dbutils.widgets.text("catalog","regis_healthcare","catalog")
dbutils.widgets.text("data_source","facilities","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
# Define S3 path
bucket = "regis-healthcare"
prefix = f"source-row-data /{data_source}"
s3_path = f"s3://{bucket}/{prefix}/"

# Use dbutils to list files and find the latest
files = dbutils.fs.ls(s3_path)
latest_file = sorted(files, key=lambda x: x.modificationTime, reverse=True)[0]

base_path = latest_file.path
print("Latest file path:", base_path)

In [0]:
df = (
    spark.read.format("csv")
       .option("header",True)
       .option("inferSchema",True)
       .load(base_path)
       .withColumn("current_date",F.current_date())
       .withColumn("read_timestamp",F.current_timestamp())
       .select("*","_metadata.file_name","_metadata.file_size")
    )
print(df.count())
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
df.write\
    .format("delta")\
            .mode("overwrite")\
                .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")
# .option("delta.enableChangeDataFeed","true")\

In [0]:
# bronze write to s3
df.write.format("delta")\
    .option("overwriteSchema","true")\
    .mode("overwrite")\
    .partitionBy("current_date")\
    .save(f"s3://regis-healthcare/bronze-row-data/{data_source}/")

#### 2. Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)
print(df_bronze.count())

In [0]:
# schema check
print(df_bronze.count())
df_bronze.printSchema()

In [0]:
df_bronze.columns

In [0]:
# drop duplicate
df_silver = df_bronze.dropDuplicates()
print(df_silver.count())

#### Silver table load


In [0]:
df_silver.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed","true")\
            .option("mergeSchema","true")\
                .option("overwriteSchema","true")\
            .mode("overwrite")\
               .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

dt = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")
print(dt.count())
display(dt)

In [0]:
# load to s3
df_silver.write.format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
    .mode("overwrite")\
    .partitionBy("current_date")\
    .save(f"s3://regis-healthcare/silver-clean-data/{data_source}/")

#### Gold Processing

In [0]:
df_silver = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")
print(df_silver.count())

In [0]:
df_gold = df_silver.select(

)
display(df_gold)

In [0]:
# df_gold.write\
#     .format("delta")\
#     .option("mergeSchema","true")\
#     .option("overwriteSchema","true")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .saveAsTable(f"{catalog}.{gold_schema}.fact_{data_source}")

In [0]:
df_gold.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"{catalog}.{gold_schema}.sb_fact_{data_source}")
print(df_gold.count())

In [0]:
df = spark.sql(f"select * from {catalog}.{gold_schema}.sb_fact_{data_source};")
print(df.count())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load Delta table with correct fully-qualified name
delta_table = DeltaTable.forName(spark, "caroucell_pro.gold.fact_admissions")
# Create DataFrame from source table with correct fully-qualified name
# sb_dim_products
df_child_products = (
    spark.table("caroucell_pro.gold.sb_fact_admissions")
    .select("*")
)
# Perform merge
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.admission_id = source.admission_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from {catalog}.{gold_schema}.fact_{data_source};")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from {catalog}.{gold_schema}.sb_fact_{data_source};")
print(sb_dim_df.count())

#### gold load to s3

In [0]:
# df_gold.write\
#     .format("delta")\
#     .option("mergeSchema","true")\
#     .option("overwriteSchema","true")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .save(f"s3://regis-healthcare/gold-delta-table/fact_{data_source}")

In [0]:
from delta.tables import DeltaTable

# ✅ Path to your Delta table stored in S3
delta_table_path = f"s3://regis-healthcare/gold-delta-table/fact_{data_source}"

# ✅ Load target Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# ✅ Source DataFrame (example: df_child_products)
source_df = df

# ✅ Perform MERGE with upsert logic
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.admission_id = source.admission_id"
    )
    .whenMatchedUpdateAll()      # Update all columns when matched
    .whenNotMatchedInsertAll()   # Insert all columns when not matched
    .execute()
)


In [0]:
display(df)